# LOC Suspicious Entity Detection — EDA

**Question**: Can we identify any suspicious provider, hospital group, or market in M&R FFS?

**Source**: `tmp_1m.kn_loc_notif_04222026_od`  
**Population**: M&R FFS (`mnr_total_ffs_flag = 1`, `loc_flag = 1`, `ipa_pac_flag in ('IPA', 'MM')`)  
**Dimensions**: Provider TIN, Hospital Group, Market (fin_market)  
**Time splits**: 2025 only (202501—202512), 2026 only (202601—202604, April partial)  
**Approach**: EDA first — profile distributions, then decide on detection method

## 1. Setup & Configuration

In [ ]:
from itables import init_notebook_mode
init_notebook_mode(all_interactive = True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')

OUTPUT_DIR = Path(r'C:\Users\knguy139\Documents\Projects\Data\Output')

SOURCE_TABLE = 'tmp_1m.kn_loc_notif_04222026_od'

YEAR_SPLITS = {
    '2025': ('202501', '202512'),
    '2026': ('202601', '202604'),  # April is partial
}

DIMENSIONS = ['prov_tin', 'hospital_group', 'fin_market']

COUNT_COLS = [
    'case_count', 'initial_adr_cnt', 'persistent_adr_cnt', 'md_reviewed_cnt',
    'appeal_case_cnt', 'appeal_ovrtn_case_cnt',
    'mcr_reconsideration_case_cnt', 'mcr_ovrtn_case_cnt',
    'p2p_case_cnt', 'p2p_ovrtn_case_cnt',
    'member_appeal_cnt', 'member_appeal_ovtn_cnt',
    'pre_auth_cases', 'membership', 'days', 'admits', 'allowed', 'netpaid',
]

MIN_CASES = 30

## 2. Snowflake Connection

In [ ]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine = create_engine(URL(
    account = 'UHG-UHGDWAAS',
    user = 'KHANG.NGUYEN@UHC.COM',
    authenticator = 'externalbrowser',
    role = 'AZU_SDRP_VING_PRD_DEVELOPER_ROLE',
    warehouse = 'VING_PRD_MNR_HCE_DATAINFRA_WH',
    database = 'VING_PRD_TREND_DB',
    schema = 'TMP_1M',
))

## 3. Data Pull

In [ ]:
df_2025 = pd.read_sql(f"""
select *
from {SOURCE_TABLE}
where ipa_pac_flag in ('IPA', 'MM')
    and loc_flag = 1
    and mnr_total_ffs_flag = 1
    and hce_admit_month between '202501' and '202512'
""", engine)

df_2025.columns = df_2025.columns.str.lower()
df_2025.shape

In [ ]:
# validation — row count, distinct members, null check on key IDs
print(f'rows: {df_2025.shape[0]:,}  distinct mbi: {df_2025["mbi"].nunique():,}')
print(df_2025[['mbi', 'prov_tin']].isnull().sum())

In [ ]:
df_2025.head()

In [ ]:
df_2025.groupby('hce_admit_month')['case_count'].agg(['count', 'sum'])

In [ ]:
df_2026 = pd.read_sql(f"""
select *
from {SOURCE_TABLE}
where ipa_pac_flag in ('IPA', 'MM')
    and loc_flag = 1
    and mnr_total_ffs_flag = 1
    and hce_admit_month between '202601' and '202604'
""", engine)

df_2026.columns = df_2026.columns.str.lower()
df_2026.shape

In [ ]:
# validation — row count, distinct members, null check on key IDs
print(f'rows: {df_2026.shape[0]:,}  distinct mbi: {df_2026["mbi"].nunique():,}')
print(df_2026[['mbi', 'prov_tin']].isnull().sum())

In [ ]:
df_2026.head()

In [ ]:
df_2026.groupby('hce_admit_month')['case_count'].agg(['count', 'sum'])

In [ ]:
engine.dispose()

## 4. Entity Profiling — Aggregate by Dimension

For each year-split and dimension, sum the count columns across months,  
then compute 14 LOC rates + 4 cost/utilization metrics.  
Volume filter: `case_count >= 30`.

In [ ]:
RATE_FEATURES = [
    'adr_rate', 'persistent_adr_rate', 'persistency', 'md_review_rate',
    'appeal_rate', 'appeal_overturn_rate', 'p2p_rate', 'p2p_overturn_rate',
    'mcr_reconsideration_rate', 'mcr_overturn_rate',
    'member_appeal_rate', 'member_appeal_overturn_rate',
    'pre_auth_rate', 'auth_per_k',
]

COST_FEATURES = ['allowed_per_case', 'netpaid_per_case', 'avg_los']

ALL_FEATURES = RATE_FEATURES + COST_FEATURES

# priority: adr/persistency first, overturns over appeals, auth_per_k over case_count
KEY_FEATURES = ['adr_rate', 'persistent_adr_rate', 'persistency',
                'appeal_overturn_rate', 'p2p_overturn_rate', 'auth_per_k']

# overturn-focused columns for dimension tables
DIM_COLS = ['adr_rate', 'persistent_adr_rate', 'persistency',
            'appeal_overturn_rate', 'p2p_overturn_rate',
            'mcr_overturn_rate', 'member_appeal_overturn_rate']

profiles = {}

for year_label, df in [('2025', df_2025), ('2026', df_2026)]:
    for dim in DIMENSIONS:
        agg = (
            df
            .groupby(dim)[COUNT_COLS]
            .sum()
            .query(f'case_count >= {MIN_CASES}')
        )
        # could be a function — compute LOC rates + cost metrics
        agg = agg.assign(
            adr_rate = lambda x: x['initial_adr_cnt'] / x['case_count'].replace(0, np.nan),
            persistent_adr_rate = lambda x: x['persistent_adr_cnt'] / x['case_count'].replace(0, np.nan),
            persistency = lambda x: x['persistent_adr_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            md_review_rate = lambda x: x['md_reviewed_cnt'] / x['case_count'].replace(0, np.nan),
            appeal_rate = lambda x: x['appeal_case_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            appeal_overturn_rate = lambda x: x['appeal_ovrtn_case_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            p2p_rate = lambda x: x['p2p_case_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            p2p_overturn_rate = lambda x: x['p2p_ovrtn_case_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            mcr_reconsideration_rate = lambda x: x['mcr_reconsideration_case_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            mcr_overturn_rate = lambda x: x['mcr_ovrtn_case_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            member_appeal_rate = lambda x: x['member_appeal_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            member_appeal_overturn_rate = lambda x: x['member_appeal_ovtn_cnt'] / x['initial_adr_cnt'].replace(0, np.nan),
            pre_auth_rate = lambda x: x['pre_auth_cases'] / x['case_count'].replace(0, np.nan),
            auth_per_k = lambda x: x['case_count'] * 1000 / x['membership'].replace(0, np.nan),
            allowed_per_case = lambda x: x['allowed'] / x['case_count'].replace(0, np.nan),
            netpaid_per_case = lambda x: x['netpaid'] / x['case_count'].replace(0, np.nan),
            avg_los = lambda x: x['days'] / x['admits'].replace(0, np.nan),
        ).reset_index()
        profiles[(year_label, dim)] = agg

# entity counts per profile
pd.DataFrame(
    [(yr, dim, len(df)) for (yr, dim), df in profiles.items()],
    columns = ['year', 'dimension', 'n_entities']
)


### Validation — 2025 prov_tin spot check

In [ ]:
profiles[('2025', 'prov_tin')][ALL_FEATURES].describe().round(4)

In [ ]:
profiles[('2025', 'prov_tin')][ALL_FEATURES].isnull().sum()

## 5. Distribution Profiling

In [ ]:
# histograms — key features across all 6 profile tables
for (year, dim), df_prof in profiles.items():
    fig, axes = plt.subplots(2, 3, figsize = (15, 8))
    fig.suptitle(f'{year} | {dim} (n={len(df_prof)})', fontsize = 14)
    for ax, feat in zip(axes.flatten(), KEY_FEATURES):
        vals = df_prof[feat].dropna()
        ax.hist(vals, bins = 50, edgecolor = 'white', alpha = 0.8)
        ax.set_title(feat)
        ax.axvline(vals.median(), color = 'red', linestyle = '--', label = f'median={vals.median():.3f}')
        ax.legend(fontsize = 8)
    plt.tight_layout()
    plt.show()

In [ ]:
# correlation heatmap — prov_tin (highest N, most representative)
for year in ['2025', '2026']:
    df_corr = profiles[(year, 'prov_tin')][ALL_FEATURES].corr()
    fig, ax = plt.subplots(figsize = (14, 10))
    sns.heatmap(df_corr, annot = True, fmt = '.2f', cmap = 'RdBu_r', center = 0,
                square = True, linewidths = 0.5, ax = ax, annot_kws = {'size': 7})
    ax.set_title(f'Feature Correlation — {year} Provider TIN', fontsize = 14)
    plt.tight_layout()
    plt.show()

In [ ]:
# skewness
(
    pd.DataFrame(
        {f'{yr}|{dim}': df_prof[ALL_FEATURES].skew().round(2)
         for (yr, dim), df_prof in profiles.items()}
    )
)

In [ ]:
# sparsity — % of entities with zero values
(
    pd.DataFrame(
        {f'{yr}|{dim}': ((df_prof[ALL_FEATURES] == 0).sum() / len(df_prof) * 100).round(1)
         for (yr, dim), df_prof in profiles.items()}
    )
)

## 6. Dimension-Specific Observations

In [ ]:
# provider TIN — auth_per_k distribution
for year in ['2025', '2026']:
    p = profiles[(year, 'prov_tin')]
    fig, axes = plt.subplots(1, 2, figsize = (14, 5))
    fig.suptitle(f'{year} Provider TIN (n={len(p)})', fontsize = 13)
    axes[0].hist(p['auth_per_k'].dropna(), bins = 50, edgecolor = 'white')
    axes[0].set_xlabel('auth_per_k')
    axes[1].hist(p['adr_rate'].dropna(), bins = 50, edgecolor = 'white', color = 'orange')
    axes[1].set_xlabel('adr_rate')
    plt.tight_layout()
    plt.show()

In [ ]:
# top 10 TINs by auth_per_k — 2025
(
    profiles[('2025', 'prov_tin')]
    .nlargest(10, 'auth_per_k')
    [['prov_tin', 'case_count', 'auth_per_k'] + DIM_COLS]
    .reset_index(drop = True)
)

In [ ]:
# top 10 TINs by auth_per_k — 2026
(
    profiles[('2026', 'prov_tin')]
    .nlargest(10, 'auth_per_k')
    [['prov_tin', 'case_count', 'auth_per_k'] + DIM_COLS]
    .reset_index(drop = True)
)

In [ ]:
# hospital group — auth_per_k quantiles
pd.DataFrame({
    yr: profiles[(yr, 'hospital_group')]['auth_per_k']
       .quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
       .round(1)
    for yr in ['2025', '2026']
})

In [ ]:
# top 10 hospital groups by auth_per_k — 2025
(
    profiles[('2025', 'hospital_group')]
    .nlargest(10, 'auth_per_k')
    [['hospital_group', 'case_count', 'auth_per_k'] + DIM_COLS]
    .reset_index(drop = True)
)

In [ ]:
# top 10 hospital groups by auth_per_k — 2026
(
    profiles[('2026', 'hospital_group')]
    .nlargest(10, 'auth_per_k')
    [['hospital_group', 'case_count', 'auth_per_k'] + DIM_COLS]
    .reset_index(drop = True)
)

In [ ]:
# market — small N, full table sorted by auth_per_k — 2025
(
    profiles[('2025', 'fin_market')]
    .sort_values('auth_per_k', ascending = False)
    [['fin_market', 'case_count', 'auth_per_k'] + DIM_COLS]
    .reset_index(drop = True)
)

In [ ]:
# market — 2026
(
    profiles[('2026', 'fin_market')]
    .sort_values('auth_per_k', ascending = False)
    [['fin_market', 'case_count', 'auth_per_k'] + DIM_COLS]
    .reset_index(drop = True)
)

## 7. Summary & Next Steps

After reviewing the EDA above, observations to note:

1. **Distribution shape**: Which features are skewed? Which have heavy zero-inflation?  
2. **Correlation clusters**: Are some rate features redundant?  
3. **Dimension suitability**: TIN has high N — standard methods work. Market has low N — may need rank-based or visual approach.  
4. **ADR/persistency signal**: Are adr_rate and persistent_adr_rate differentiating entities?  
5. **Overturn patterns**: Which overturn rates (appeal, p2p, mcr, member_appeal) show the most variance?  

**Detection method candidates** (to decide after reviewing EDA output):  
- Percentile + IQR flagging (simple, explainable)  
- Robust z-score (MAD-based — better for skewed distributions)  
- Isolation Forest (multivariate — less explainable)  